# Notebook 2: Audio Feature Clustering (Unsupervised Learning)
**Objective:** Transform the clean audio features into a mathematical model. We will use **Principal Component Analysis (PCA)** to reduce noise and compress the data, followed by **K-Means Clustering** to organize the 113,000 tracks into distinct, mathematically similar "Vibe Galaxies" without relying on human-assigned genres.

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Load the clean dataset from Notebook 1
df_clean = pd.read_csv('data/cleaned_spotify_tracks.csv')

# Define the exact features we want the AI to learn from
features_to_use = [
    'danceability', 'energy', 'loudness', 'speechiness', 
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

### 1. Feature Scaling & Dimensionality Reduction (PCA)
Distance-based Machine Learning algorithms (like K-Means) are highly sensitive to scale. Because `tempo` ranges from 0 to 200, but `acousticness` ranges from 0 to 1, the Clustering Algorithm will naturally assume tempo is 200x more important. We use a **StandardScaler** to force all features onto an equal playing field.

Furthermore, audio features often overlap (e.g., loudness and energy). We deploy **Principal Component Analysis (PCA)** to compress these 9 features into 4 "Super Features," stripping away mathematical noise and massively optimizing the search speed of our final engine.

In [2]:
# 1. Scale the Data
print("Scaling features...")
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_clean[features_to_use])

# 2. Apply PCA (Principal Component Analysis)
print("Running PCA...")
pca = PCA(n_components=4, random_state=42)
pca_features = pca.fit_transform(scaled_features)

# Calculate how much of the original "story" we kept (Variance)
variance_kept = sum(pca.explained_variance_ratio_) * 100
print(f"-> PCA compressed 9 features into 4, preserving {variance_kept:.1f}% of the original variance!")

Scaling features...
Running PCA...
-> PCA compressed 9 features into 4, preserving 71.4% of the original variance!


### 2. Grouping the Universe (K-Means Clustering)
Now that our data is scaled and compressed, we deploy **K-Means Clustering** to divide the 113,000 songs into 15 distinct clusters. 

Because this is *Unsupervised Learning*, the Unsupervised Model does not know what a "Guitar" or a "Synthesizer" is. It is simply plotting these tracks on a 4-dimensional map and drawing borders around the groups that are mathematically clustered together.

In [ ]:
# Group the dataset into 15 distinct "Vibe Clusters"
print("Running KMeans Clustering (k=15)...")

# n_init=10 ensures the algorithm runs multiple times to find the absolute best center points
kmeans = KMeans(n_clusters=15, random_state=42, n_init=10)
kmeans.fit(pca_features)

# Save the ML model's grouping decisions back into our dataframe
df_clean['cluster_id'] = kmeans.labels_

print("Cluster Population Breakdown:")
print(df_clean['cluster_id'].value_counts().sort_index())

Running KMeans Clustering (k=15)...
Cluster Population Breakdown:
cluster_id
0     10984
1      6094
2      3187
3      5835
4      8400
5     12289
6      9984
7     15434
8      6572
9      5601
10      914
11     9207
12     9489
13     4967
14     5042
Name: count, dtype: int64


### 3. Cracking the "Black Box" (Cluster Profiling)
A common issue in unsupervised learning is the "Black Box" problem. The clustering algorithm successfully grouped the tracks into 15 clusters (numbered 0 through 14), but it has no vocabulary to explain why it grouped them together.

To turn this raw math into actual insights, we must profile these clusters and assign them human meaning. Below, we loop through the model's buckets, calculating the average audio features of the bucket and inspecting the most popular tracks inside it.

In [4]:
print("--- CRACKING OPEN THE CLUSTERS ---")

# We only want to look at the features that are easy for humans to understand
human_features = ['danceability', 'energy', 'acousticness', 'valence', 'tempo']

# Calculate the average (mean) of every feature for each cluster
cluster_averages = df_clean.groupby('cluster_id')[human_features].mean()

# Loop through all 15 clusters to print their "Profile"
for cluster_num in range(15):
    print(f"🔍 CLUSTER {cluster_num}")
    
    # Get the average stats for this specific cluster
    stats = cluster_averages.loc[cluster_num]
    
    # Find the top 3 most common genres in this cluster
    top_genres = df_clean[df_clean['cluster_id'] == cluster_num]['track_genre'].value_counts().head(3).index.tolist()
    
    # Find 3 highly popular songs as examples and glue the artist name to the track
    top_songs = df_clean[df_clean['cluster_id'] == cluster_num].sort_values(by='popularity', ascending=False).drop_duplicates('track_name').head(3)
    combined_names = top_songs['track_name'] + " (by " + top_songs['artists'] + ")"
    song_names = combined_names.tolist()
    
    # Print the profile!
    print(f"   Vibe:   Energy: {stats['energy']:.2f} | Dance: {stats['danceability']:.2f} | Acoustic: {stats['acousticness']:.2f} | Happiness: {stats['valence']:.2f}")
    print(f"   Genres: {', '.join(top_genres).title()}")
    print(f"   Tracks: {', '.join(song_names)}")
    print("-" * 75)

--- CRACKING OPEN THE CLUSTERS ---
🔍 CLUSTER 0
   Vibe:   Energy: 0.80 | Dance: 0.62 | Acoustic: 0.18 | Happiness: 0.74
   Genres: Forro, Party, J-Idol
   Tracks: I Ain't Worried (by OneRepublic), As It Was (by Harry Styles), La Corriente (by Bad Bunny;Tony Dize)
---------------------------------------------------------------------------
🔍 CLUSTER 1
   Vibe:   Energy: 0.68 | Dance: 0.66 | Acoustic: 0.10 | Happiness: 0.31
   Genres: Minimal-Techno, Detroit-Techno, Techno
   Tracks: Apocalypse (by Cigarettes After Sex), Alien Blues (by Vundabar), Chamber Of Reflection (by Mac DeMarco)
---------------------------------------------------------------------------
🔍 CLUSTER 2
   Vibe:   Energy: 0.57 | Dance: 0.52 | Acoustic: 0.60 | Happiness: 0.52
   Genres: Pagode, Samba, Mpb
   Tracks: Pink + White (by Frank Ocean), Love Yourself (by Justin Bieber), MONTERO (Call Me By Your Name) (by Lil Nas X)
---------------------------------------------------------------------------
🔍 CLUSTER 3
   Vibe: 

### 💡 Key Analyst Observations
By profiling the clusters, we can confidently name the AI's groupings:
* The "Do Not Play at a Party" Bucket (Cluster 9): The clustering algorithm perfectly isolated ambient, sleep, and new-age music, calculating a massive Acousticness score of 0.87 and a near-zero Energy score.
* The "Mosh Pit" Bucket (Cluster 13): Grindcore, Black-Metal, and Death-Metal were successfully isolated based purely on their maxed-out Energy (0.87) and incredibly low Happiness/Valence (0.17).
* The Stand-Up Comedy Quarantine (Cluster 10): A tiny cluster (914 tracks) containing Comedy and Kids audio. To the unsupervised model, a stand-up comedian speaking into a microphone mathematically registers as high-energy and highly acoustic (due to the lack of instruments). The estimator recognized this mathematical anomaly and successfully quarantined it from the actual music tracks!
### ⚠️ Architecture Pivot Note (Setup for Notebook 3)
While this local PCA & K-Means model is highly accurate, it relies entirely on the 9 specific audio features provided by Spotify. Because Spotify recently deprecated their developer API for audio features, we cannot pass brand-new, live tracks into this specific model (as we cannot retrieve their danceability or energy).

In Notebook 3, we will pivot the final recommendation engine to use an NLP architecture (TF-IDF) utilizing live, crowd-sourced tags from the Last.fm API to bypass this limitation.